In [1]:
#------IMPORTING LIBRARIES & OHLC DATA------

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

df = yf.download("AMZN", start="2005-01-01", auto_adjust=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

breakout = 20
atr_mult = 3.5

[*********************100%***********************]  1 of 1 completed


In [2]:
#------STRATEGY------

def run_backtest(df, breakout, atr_mult, slip_factor = 1.0):
    df = df.copy()
    
    df['breakout_high'] = df['High'].shift(1).rolling(breakout).max()
    df['breakout_low'] = df['Low'].shift(1).rolling(breakout).min()
    df['sma200'] = df['Close'].rolling(200).mean()
    df['returns_std'] = df['Close'].pct_change(fill_method=None).rolling(breakout).std()
    df['true_range'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            abs(df['High'] - df['Close'].shift(1)),
            abs(df['Low'] - df['Close'].shift(1))
        )
    )
    df['atr'] = df['true_range'].rolling(14).mean()
    df['returns_std_ma'] = df['returns_std'].rolling(100).mean()
    
    capital = 100000
    cash = capital
    cost = 0.001
    position = 0
    entry_price = 0
    highest_price = None

    position_size = 0.7     #best out of 0.5-1.0
    
    equity = []
    
    trades = []
    
    entry_idx = None
    
    position_history = []
    
    
    for i in range(len(df)):

        close = df['Close'].iloc[i]
        
        
        if i == len(df) - 1:
            equity.append(cash + position * close)
            position_history.append(1 if position > 0 else 0)
            break
            
        next_open = df['Open'].iloc[i+1]
        
        breakout_high = df['breakout_high'].iloc[i]
        breakout_low  = df['breakout_low'].iloc[i]
        sma = df['sma200'].iloc[i]
        returns_std = df['returns_std'].iloc[i]
        returns_std_ma = df['returns_std_ma'].iloc[i]
        atr = df['atr'].iloc[i]
        
        # atr-based slippage
        
        slippage = (0.1 * atr / close) * slip_factor 
        
        
        #NaN handling
        if any(pd.isna(x) for x in [breakout_high, breakout_low, sma, returns_std, returns_std_ma, atr]):
            equity.append(cash + position * close)
            position_history.append(1 if position > 0 else 0)
            continue

            
        # Entry:
        
        if position == 0:
    
            if close > breakout_high:
    
                entry_price = next_open * (1 + cost/2 + slippage)
                shares = (cash * position_size) / entry_price
                
                position = shares
                cash -= shares * entry_price
                
                highest_price = close
                entry_idx = i # Real cash deduction model
                
    
        # Exit:
        
        elif position > 0:
            
            
            if highest_price is None:
                highest_price = close
            else:
                highest_price = max(highest_price, close)
            
            atr_stop = highest_price - atr_mult * atr
            donchian_stop = breakout_low
            
            exit_level = max(donchian_stop, atr_stop)
    
            
            # ATR-based exit (replace low20 exit)
            if close <  exit_level:
                exit_price = next_open * (1 - cost/2 - slippage)
               
                pnl = (exit_price - entry_price) / entry_price
                duration = i - entry_idx
                trades.append((pnl, duration)) 
    
                cash += position * exit_price
                position = 0
                entry_price = 0
                entry_idx = None
                highest_price = None
                
    
        # Equity
        
        equity.append(cash + position * close)
        position_history.append(1 if position > 0 else 0)

    
        #  Metrics
    
    equity = pd.Series(equity, index=df.index)
    
    # Returns
    
    returns = equity.pct_change()
    
    # CAGR (time-based):
    
    years = (df.index[-1] - df.index[0]).days / 365.25
    
    if years > 0:
        cagr = (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1
    else:
        cagr = np.nan
    
    # Vol / Sharpe:
    
    if returns.std() > 0:
        vol_ann = returns.std() * np.sqrt(252)
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
    else:
        vol_ann = sharpe = np.nan
        
    # Sortino:
    
    downside = returns[returns < 0]
    sortino = (returns.mean() / downside.std()) * np.sqrt(252) if downside.std() > 0 else np.nan

    downside = returns[returns < 0]

    if len(downside) > 0 and downside.std() > 0:
        sortino = (returns.mean() / downside.std()) * np.sqrt(252)
    else:
        sortino = np.nan
        
    # Drawdowns:
    
    cum_max = equity.cummax()
    drawdown = equity / cum_max - 1
    
    max_dd = drawdown.min()
    avg_dd = drawdown[drawdown < 0].mean()
    
    # Drawdown Duration:
    
    dd_flag = (drawdown < 0).astype(int)
    
    dd_duration = (
        dd_flag.groupby((dd_flag != dd_flag.shift()).cumsum())
        .cumsum()
    )
    
    max_dd_duration = dd_duration.max()
    avg_dd_duration = dd_duration[dd_duration > 0].mean()
    
    # Trade Stats:
    
    if len(trades) > 0:
        trade_returns = np.array([t[0] for t in trades])
        trade_durations = np.array([t[1] for t in trades])
    
        win_rate = (trade_returns > 0).mean()
    
        wins = trade_returns[trade_returns > 0]
        losses = trade_returns[trade_returns < 0]
    
        profit_factor = (
            wins.sum() / abs(losses.sum())
            if len(losses) > 0 else np.nan
        )
    
        expectancy = trade_returns.mean()
        best_trade = trade_returns.max()
        worst_trade = trade_returns.min()
    
        max_trade_duration = trade_durations.max()
        avg_trade_duration = trade_durations.mean()
    
        # SQN:
        
        if len(trade_returns) > 1 and trade_returns.std() > 0:
            sqn = (trade_returns.mean() / trade_returns.std()) * np.sqrt(len(trade_returns))
        else:
            sqn = np.nan
    
        # Kelly:
        
        if len(wins) > 0 and len(losses) > 0:
            kelly = win_rate - (1 - win_rate) / (wins.mean() / abs(losses.mean()))
        else:
            kelly = np.nan
    
    else:
        win_rate = profit_factor = expectancy = np.nan
        best_trade = worst_trade = np.nan
        max_trade_duration = avg_trade_duration = sqn = kelly = np.nan
        
    
    # Exposure:
    
    exposure = np.mean(position_history)
    
    # Buy & Hold:
    
    buy_hold = df['Close'].iloc[-1] / df['Close'].iloc[0] - 1
    
    #  Printing result metrics

    return {
        "cagr": cagr,
        "sharpe": sharpe,
        "sortino": sortino,
        "vol_ann": vol_ann,
        "max_dd": max_dd,
        "avg_dd": avg_dd,

        
        "max_dd_duration": max_dd_duration,
        "avg_dd_duration": avg_dd_duration,
        
        "trades": trades,
        "win_rate": win_rate,
        "profit_factor": profit_factor,
        "expectancy": expectancy,
        
        "best_trade": best_trade,
        "worst_trade": worst_trade,
        "sqn": sqn,
        "kelly": kelly,
        
        "exposure": exposure,
        "buy_hold": buy_hold,

        "equity_final": equity.iloc[-1]
    }

In [3]:
#------EXECUTION ENGINE------

class LiveEngine:

    def __init__(self, breakout, atr_mult, initial_capital=100000, slip_factor=1.0, cost=0.001, position_size=0.7):

        self.breakout = breakout
        self.atr_mult = atr_mult

        self.df = pd.DataFrame()

        self.cash = initial_capital
        self.position = 0          # number of shares
        self.entry_price = None
        self.entry_idx = None
        self.highest_price = None

        self.slip_factor = slip_factor
        self.cost = cost
        self.position_size = position_size

        self.pending_order = None  # <-- key: delay execution to next bar

        self.trades = []
        self.equity_curve = []

    def on_bar(self, new_row):

        # appended new data
        
        self.df = pd.concat([self.df, new_row.to_frame().T])

        df = self.df.copy()

        # need enough data
        
        if len(df) < 200:
            return None

        #  Indicators (same as backtest)
        
        df['breakout_high'] = df['High'].shift(1).rolling(self.breakout).max()
        df['breakout_low'] = df['Low'].shift(1).rolling(self.breakout).min()

        df['true_range'] = np.maximum(
            df['High'] - df['Low'],
            np.maximum(
                abs(df['High'] - df['Close'].shift(1)),
                abs(df['Low'] - df['Close'].shift(1))
            )
        )
        df['atr'] = df['true_range'].rolling(14).mean()

        i = len(df) - 1

        close = df['Close'].iloc[i]
        signal_close = df['Close'].iloc[i-1]

        
        breakout_high = df['breakout_high'].iloc[i]
        breakout_low = df['breakout_low'].iloc[i]
        
        atr = df['atr'].iloc[i]
        signal_atr   = df['atr'].iloc[i-1]
        

        # Executing Pending Order (at this bar open)
        
        if self.pending_order is not None:

            open_price = df['Open'].iloc[i]
            
            slippage = (0.1 * signal_atr / signal_close) * self.slip_factor
            

            if self.pending_order == "buy":
                entry_price = open_price * (1 + self.cost/2 + slippage)

                shares = (self.cash * self.position_size) / entry_price
                self.position = shares
                self.cash -= shares * entry_price

                self.entry_price = entry_price
                self.entry_idx = i
                self.highest_price = close

            elif self.pending_order == "sell":
                exit_price = open_price * (1 - self.cost/2 - slippage)

                pnl = (exit_price - self.entry_price) / self.entry_price
                duration = i - self.entry_idx

                self.trades.append((pnl, duration))

                self.cash += self.position * exit_price
                self.position = 0
                self.entry_price = None
                self.entry_idx = None
                self.highest_price = None

            self.pending_order = None
            

        #  Updating Trailing Logic
        
        if self.position > 0:
            self.highest_price = max(self.highest_price, close)

            atr_stop = self.highest_price - self.atr_mult * atr
            exit_level = max(breakout_low, atr_stop)

            if close < exit_level:
                self.pending_order = "sell"

        # Entry Signal
        
        if self.position == 0:
            if close > breakout_high:
                self.pending_order = "buy"

        # Equity Tracking
        
        equity = self.cash + self.position * close
        self.equity_curve.append(equity)

        return {
            "time": df.index[-1],
            "price": close,
            "position": self.position,
            "cash": self.cash,
            "equity": equity
        }

In [4]:
#------BACKTEST VS LIVE ENGINE VALIDATION------

engine = LiveEngine(breakout=20, atr_mult=3.5)

logs = []

for i in range(len(df)):

    out = engine.on_bar(df.iloc[i])

    if out:

        logs.append(out)

paper_df = pd.DataFrame(logs)

In [5]:
#  Printing results

print("Backtest equity:", run_backtest(df.copy(), 20, 3.5)['equity_final'])

print("Live sim equity:", engine.equity_curve[-1])

Backtest equity: 808925.3170974902
Live sim equity: 808925.3170974902


In [6]:
#  Live Engine Validation Export (CSV)

validation_df = pd.DataFrame({

    "metric": [
        "backtest_equity",
        "live_sim_equity",
        "difference"
    ],

    "value": [

        run_backtest(df.copy(), 20, 3.5)["equity_final"],

        engine.equity_curve[-1],

        engine.equity_curve[-1]
        - run_backtest(df.copy(), 20, 3.5)["equity_final"]
    ]
})

validation_df.to_csv(
    "results/live_execution_engine/live_engine_validation.csv",
    index=False
)

In [7]:
#  Live Execution Engine Validation Summary:

backtest_eq = run_backtest(df.copy(), 20, 3.5)["equity_final"]

live_eq = engine.equity_curve[-1]

summary = f"""
# Live Execution Engine Validation

Backtest Equity: {backtest_eq:.2f}

Live Simulation Equity: {live_eq:.2f}

Difference: {(live_eq-backtest_eq):.8f}

Status:
{"PASS" if abs(live_eq-backtest_eq) < 1e-6 else "FAIL"}

"""

with open(
    "results/live_execution_engine/summary_liveengine.md",
    "w"
) as f:

    f.write(summary)

In [8]:
#  Live Simulation Engine Equity Curve Plot:

plt.figure(figsize=(12,6))

plt.plot(engine.equity_curve)

plt.title("Live Simulation Equity Curve")

plt.ylabel("Equity")

plt.savefig(

    "results/live_execution_engine/live_equity_curve.png",

    bbox_inches="tight"

)

plt.close()